# Discesa stocastica del gradiente



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Addestrerai una rete sul dataset **Fuel Economy** ed esplorerai gli effetti di learning rate e batch size. Esegui la cella di configurazione.


In [ ]:
# Configura i grafici.
import matplotlib.pyplot as plt
from learntools.deep_learning_intro.dltools import animate_sgd
plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('animation', html='html5')

# Configura il sistema di verifica degli esercizi.
from learntools.core import binder
binder.bind(globals())
from learntools.deep_learning_intro.ex3 import *

Il compito è prevedere l'efficienza nei consumi di un'automobile da caratteristiche come motore e anno di produzione. Carica i dati con la cella seguente.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.model_selection import train_test_split

fuel = pd.read_csv('../input/dl-course-data/fuel.csv')

X = fuel.copy()
# Rimuovi il target.
y = X.pop('FE')

preprocessor = make_column_transformer(
    (StandardScaler(),
     make_column_selector(dtype_include=np.number)),
    (OneHotEncoder(sparse=False),
     make_column_selector(dtype_include=object)),
)

X = preprocessor.fit_transform(X)
y = np.log(y) # Applica il logaritmo al target invece della standardizzazione.

input_shape = [X.shape[1]]
print("Input shape: {}".format(input_shape))

Esamina i dati. Il target è **`FE`**; le altre colonne sono le feature.


In [ ]:
# Rimuovi il simbolo di commento per vedere i dati originali.
fuel.head()
# Rimuovi il simbolo di commento per vedere le feature elaborate.
pd.DataFrame(X[:10,:]).head()

Esegui la cella per definire la rete proposta.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=input_shape),
    layers.Dense(128, activation='relu'),    
    layers.Dense(64, activation='relu'),
    layers.Dense(1),
])

# 1. Loss e optimizer
Usa **`compile`** per aggiungere **Adam** e la loss **MAE**.


In [ ]:
# SCRIVI QUI IL CODICE
____

# Controlla la risposta.
q_1.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_1.hint()
#q_1.solution()

# 2. Addestrare il modello
Dopo la compilazione, addestra la rete per **200 epoche**, con **batch size 128**. Gli ingressi sono **`X`**, il target **`y`**.


In [ ]:
# SCRIVI QUI IL CODICE
history = ____

# Controlla la risposta.
q_2.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_2.hint()
#q_2.solution()

Esegui la cella seguente per disegnare la curva della training loss e valutare l'addestramento.


In [ ]:
import pandas as pd

history_df = pd.DataFrame(history.history)
# Inizia il grafico dall'epoca 5; puoi cambiare questo valore per modificare la visualizzazione.
history_df.loc[5:, ['loss']].plot();

# 3. Valutare il training
Ti aspetti che aumentando ancora le epoche la loss diminuisca? Motiva leggendo la curva.


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_3.check()

Learning rate e batch size influenzano durata del training, oscillazioni delle curve e loss raggiunta. Per studiarli usiamo un modello con un peso e un bias, più facile da interpretare.

La cella genera un'animazione simile a quella del tutorial. Modifica **`learning_rate`**, **`batch_size`** e **`num_examples`**, poi esegui. Potrebbe richiedere un po' di tempo. Prova queste combinazioni o scegline altre:

| learning_rate | batch_size | num_examples |
| --- | --- | --- |
| 0.05 | 32 | 256 |
| 0.05 | 2 | 256 |
| 0.05 | 128 | 256 |
| 0.02 | 32 | 256 |
| 0.2 | 32 | 256 |
| 1.0 | 32 | 256 |
| 0.9 | 4096 | 8192 |
| 0.99 | 4096 | 8192 |


In [ ]:
# SCRIVI QUI IL CODICE: prova valori diversi di learning rate, batch size e numero di esempi.
learning_rate = 0.05
batch_size = 32
num_examples = 256

animate_sgd(
    learning_rate=learning_rate,
    batch_size=batch_size,
    num_examples=num_examples,
    # Puoi modificare anche questi valori.
    steps=50, # Numero totale di passi di training, cioè batch elaborati.
    true_w=3.0, # Pendenza dei dati.
    true_b=2.0, # Termine costante dei dati.
)

# 4. Learning rate e batch size
Quali effetti hai osservato? Ragiona prima di leggere il commento proposto dalla cella successiva.


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_4.check()

Impara a **[migliorare le prestazioni](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/tut4.html)** modificando la capacità e aggiungendo early stopping.
